In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize()

In [ ]:
# Create an interactive map and draw roi
Map = geemap.Map()
Map

In [ ]:
# read last drawn roi as ee featurecollection
roi = ee.FeatureCollection(Map.draw_last_feature)

In [ ]:
# Load and filter Landsat 8/9 imagery
dataset = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")  # Use "LANDSAT/LC09/C02/T1_L2" for Landsat 9
           .filterBounds(roi)
           .filterDate('2024-01-01', '2024-12-31')  # Set your date range
           .filter(ee.Filter.lt('CLOUD_COVER', 10))) # Filter for low cloud cover

# Create a median composite image
image = dataset.median().clip(roi)

# Calculate NDWI
# NDWI = (Green - NIR) / (Green + NIR)
# For Landsat 8/9: Green = 'SR_B3', NIR = 'SR_B5'
ndwi = image.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI')

#  Visualize NDWI on the map
ndwi_vis = {'min': -0.5, 'max': 0.5, 'palette': ['red', 'white', 'blue']}
Map.addLayer(ndwi, ndwi_vis, 'NDWI')
Map

In [ ]:
# Threshold the NDWI to create a binary water mask
# A common threshold is 0.2, but you may need to adjust it.
water_mask = ndwi.gt(0).selfMask()  # .gt(0) creates a binary image (1 for water, 0 for non-water)
                                        # .selfMask() keeps only the water pixels

# Convert the binary mask to a vector polygon
# First, reduce the image to a single band, then reduce to vectors
vectors = water_mask.reduceToVectors(
    geometry=roi,
    geometryType='polygon',  # Output as polygons
    scale=30,                # Landsat resolution is 30m
    eightConnected=False,
    maxPixels=1e13
)

# Add the vector layer to the map
Map.addLayer(vectors, {'color': 'blue'}, 'Waterbody Vector')
Map

In [ ]:
# 8. Export the vector to a shapefile
# This will save the file to your current working directory.
geemap.ee_to_shp(vectors, filename='C:/Users/user/Downloads/waterbody.shp')